In [5]:
import os
import shutil
import pandas as pd

# Set iteration of study and source path
study = 'Study5.0Pilot3'
condition = 'Predict'
source_dir = "/Users/sm6511/Downloads/pre_6/"

dest_dir = (
    f"/Users/sm6511/Desktop/Prediction-Accomodation-Exp/"
    f"data/{study}/{condition}"
)

target_dates = [
    "2026-05-30"
]

DELETE_FAILED_ATTENTION = False

completion_col = "button_end.numClicks"

if condition.lower() == "predict":
    attention_col = "answer_3_right.numClicks"
elif condition.lower() == "accommodate":
    attention_col = "button_3_correct.numClicks"
else:
    raise ValueError("condition must be 'Predict' or 'Accommodate'")

os.makedirs(dest_dir, exist_ok=True)

moved = []
skipped = []
deleted_failed_attention = []

# collect files sorted by earlier date
candidate_files = sorted([
    fname for fname in os.listdir(source_dir)
    if fname.endswith(".csv") and any(d in fname for d in target_dates)
])

# STEP 1: delete failed attention checks
if DELETE_FAILED_ATTENTION:
    for fname in candidate_files:
        src_path = os.path.join(source_dir, fname)

        try:
            df = pd.read_csv(src_path)
            df.columns = df.columns.str.strip()
        except Exception as e:
            print(f"Could not read {fname}: {e}")
            skipped.append((fname, "read error during attention check"))
            continue

        if attention_col not in df.columns:
            print(f"Missing {attention_col} in {fname}")
            skipped.append((fname, "missing attention column"))
            continue

        passed_attention = (
            pd.to_numeric(df[attention_col], errors="coerce")
            .eq(1)
            .any()
        )

        if not passed_attention:
            print(f"🗑️ DELETING FAILED ATTENTION CHECK: {fname}")
            os.remove(src_path)
            deleted_failed_attention.append(fname)

# refresh list of files
candidate_files = sorted([
    fname for fname in os.listdir(source_dir)
    if fname.endswith(".csv") and any(d in fname for d in target_dates)
])

# STEP 2: move completed files, earliest first
for fname in candidate_files:
    src_path = os.path.join(source_dir, fname)

    try:
        df = pd.read_csv(src_path)
        df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"Could not read {fname}: {e}")
        skipped.append((fname, "read error"))
        continue

    if completion_col not in df.columns:
        print(f"Missing {completion_col} in {fname}")
        skipped.append((fname, "missing completion column"))
        continue

    is_complete = (
        pd.to_numeric(df[completion_col], errors="coerce")
        .eq(1)
        .any()
    )

    if is_complete:
        dest_path = os.path.join(dest_dir, fname)
        shutil.move(src_path, dest_path)
        moved.append(fname)
        print(f"MOVED: {fname}")
    else:
        skipped.append((fname, f"{completion_col} != 1"))

print("\n===== SUMMARY =====")

print(f"Deleted failed attention files ({len(deleted_failed_attention)}):")
for f in deleted_failed_attention:
    print(f"  {f}")

print(f"\nMoved files ({len(moved)}):")
for f in moved:
    print(f"  {f}")

print(f"\nSkipped files ({len(skipped)}):")
for f, reason in skipped:
    print(f"  {f} — {reason}")

MOVED: 001_test_2026-05-30_17h39.25.482.csv
MOVED: 002_test_2026-05-30_15h21.00.802.csv
MOVED: 003_test_2026-05-30_16h21.06.466.csv
MOVED: 004_test_2026-05-30_17h21.07.678.csv
MOVED: 005_test_2026-05-30_16h39.55.780.csv
MOVED: 006_test_2026-05-30_17h21.27.280.csv
MOVED: 007_test_2026-05-30_16h42.24.377.csv
MOVED: 008_test_2026-05-30_16h22.35.156.csv
MOVED: 009_test_2026-05-30_17h22.57.801.csv
MOVED: 010_test_2026-05-30_16h23.32.455.csv
MOVED: 011_test_2026-05-30_17h43.17.362.csv
MOVED: 012_test_2026-05-30_17h23.22.894.csv
MOVED: 013_test_2026-05-30_17h23.19.632.csv
MOVED: 014_test_2026-05-30_17h23.23.256.csv
MOVED: 015_test_2026-05-30_17h23.17.299.csv
MOVED: 016_test_2026-05-30_17h23.51.159.csv
MOVED: 017_test_2026-05-30_17h23.31.883.csv
MOVED: 018_test_2026-05-30_17h23.56.566.csv
MOVED: 019_test_2026-05-30_14h25.34.298.csv
MOVED: 020_test_2026-05-30_14h28.51.015.csv
Missing button_end.numClicks in 021_test_2026-05-30_14h25.54.019.csv
MOVED: 021_test_2026-05-30_17h51.49.067.csv
MOVED: 